# Importing the required libraries

### Importing and Pandas configurations
Here we are importing the os and glob to read multiple datafiles to python code/env
Also setting up the pandas df display settings

In [1]:
import os
import glob
import pandas as pd
import numpy as np
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

### Gathering the data
Gathering all the csv files from the given folder path and storing them in all files


In [2]:
#Grab all CSV paths from your target folder
folder_path = r"X:\Powerbi\Projects\Brazillian_ecommerce_analysis\Data"
all_files = glob.glob(os.path.join(folder_path, "*.csv"))


# Extracting the Structural data and Health profiles

### Defining the required dicts and lists and extracting data from files
Here the every csv dat is extracted using the for loop and storing the data files as pd dataframes in dict and lists the diffrent dictionaries are created for the different profiling you will see those in below cells


In [3]:
# 2. Extract structural data health profiles
file_profiles =[]
dfs =[]
column_def=[]
describe_prf=[]
null_count=[]
samples_f=[]
for file in all_files:
    try:
        # Load sample or full file depending on size
        df_temp = pd.read_csv(file) # Quick check on first 1000 rows with nrows=1000
        
        profile = {
            "File_Name": os.path.basename(file),
            "Row_Count": len(df_temp),
            "Column_Count": len(df_temp.columns),
            #"Columns": list(df_temp.columns),
            "Missing_Values_Total": df_temp.isnull().sum().sum()
        }
        column_prf={
            "File_Name": os.path.basename(file),
            "Column names": list(df_temp.columns)
        }
        nullcntf={
            "File_Name": os.path.basename(file),
            "Null columns": df_temp.isnull().sum()
        }
        des_prf={
            "File_Name": os.path.basename(file),
            "Describe": df_temp.describe(include="all")
        }
        samples={
            "File_Name": os.path.basename(file),
            "Samples": df_temp.sample(10)
        }

        
        file_profiles.append(profile)
        column_def.append(column_prf)
        null_count.append(nullcntf)
        describe_prf.append(des_prf)
        samples_f.append(samples)

        # Keep track for final master merge
        #dfs.append(pd.read_csv(file))
    except Exception as e:
        print(f"⚠️ Structural error found in file {os.path.basename(file)}: {e}")

### Profiling the shape and missing values
Here you can see the total No. of rows and columns along with the missing values of those files

In [4]:
# Validate if all files match column layouts
profile_df = pd.DataFrame(file_profiles)
profile_df

# 3. If clean, consolidate into a master staging dataframe
#if len(dfs) > 0:
#    master_df = pd.concat(dfs, ignore_index=True)
#    # Save a staging dataset
#    master_df.to_csv(os.path.join(folder_path, "master_staging.csv"), index=False)

,File_Name,Row_Count,Column_Count,Missing_Values_Total
0,olist_customers_dataset.csv,99441,5,0
1,olist_geolocation_dataset.csv,1000163,5,0
2,olist_orders_dataset.csv,99441,8,4908
3,olist_order_items_dataset.csv,112650,7,0
4,olist_order_payments_dataset.csv,103886,5,0
5,olist_order_reviews_dataset.csv,99224,7,145903
6,olist_products_dataset.csv,32951,9,2448
7,olist_sellers_dataset.csv,3095,4,0
8,product_category_name_translation.csv,71,2,0


### Verifyin columns
Here you can check if all the csv files are haivng the required columns as expected

In [5]:
column_df=pd.DataFrame(column_def)
column_df

,File_Name,Column names
0,olist_customers_dataset.csv,"[customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state]"
1,olist_geolocation_dataset.csv,"[geolocation_zip_code_prefix, geolocation_lat, geolocation_lng, geolocation_city, geolocation_state]"
2,olist_orders_dataset.csv,"[order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date]"
3,olist_order_items_dataset.csv,"[order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value]"
4,olist_order_payments_dataset.csv,"[order_id, payment_sequential, payment_type, payment_installments, payment_value]"
5,olist_order_reviews_dataset.csv,"[review_id, order_id, review_score, review_comment_title, review_comment_message, review_creation_date, review_answer_timestamp]"
6,olist_products_dataset.csv,"[product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm]"
7,olist_sellers_dataset.csv,"[seller_id, seller_zip_code_prefix, seller_city, seller_state]"
8,product_category_name_translation.csv,"[product_category_name, product_category_name_english]"


### Null values ?
Here we can find where the actual null values exists in those files

In [6]:
for item in null_count:
    s = item["Null columns"]
    s = s[s > 0]

    if s.empty:
        pass
        #print(item["File_Name"])
        #print("No null values\n")
    else:
        print(item["File_Name"])
        display(s.to_frame("null_count"))
        #print(s)

olist_orders_dataset.csv


,null_count
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965


olist_order_reviews_dataset.csv


,null_count
review_comment_title,87656
review_comment_message,58247


olist_products_dataset.csv


,null_count
product_category_name,610
product_name_lenght,610
product_description_lenght,610
product_photos_qty,610
product_weight_g,2
product_length_cm,2
product_height_cm,2
product_width_cm,2


### Describing data
Here we describe the data to see if there are any out right anomalies that are need to be questioned

In [7]:
for item in describe_prf:
  print(item["File_Name"])
  display(item["Describe"])

olist_customers_dataset.csv


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
count,99441,99441,99441.000000,99441,99441
unique,99441,96096,NaN,4119,27
top,06b8999e2fba1a1fbc88172c00ba8bc7,8d50f5eadf50201ccdcedfb9e2ac8455,NaN,sao paulo,SP
freq,1,17,NaN,15540,41746
mean,NaN,NaN,35137.474583,NaN,NaN
std,NaN,NaN,29797.938996,NaN,NaN
min,NaN,NaN,1003.000000,NaN,NaN
25%,NaN,NaN,11347.000000,NaN,NaN
50%,NaN,NaN,24416.000000,NaN,NaN
75%,NaN,NaN,58900.000000,NaN,NaN


olist_geolocation_dataset.csv


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
count,1.000163e+06,1.000163e+06,1.000163e+06,1000163,1000163
unique,NaN,NaN,NaN,8011,27
top,NaN,NaN,NaN,sao paulo,SP
freq,NaN,NaN,NaN,135800,404268
mean,3.657417e+04,-2.117615e+01,-4.639054e+01,NaN,NaN
std,3.054934e+04,5.715866e+00,4.269748e+00,NaN,NaN
min,1.001000e+03,-3.660537e+01,-1.014668e+02,NaN,NaN
25%,1.107500e+04,-2.360355e+01,-4.857317e+01,NaN,NaN
50%,2.653000e+04,-2.291938e+01,-4.663788e+01,NaN,NaN
75%,6.350400e+04,-1.997962e+01,-4.376771e+01,NaN,NaN


olist_orders_dataset.csv


...

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32951,32341,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
unique,32951,73,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,1e9e8ef04dbcff4541ed26657ea517e5,cama_mesa_banho,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,3029,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,NaN,NaN,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,NaN,NaN,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,NaN,NaN,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,NaN,NaN,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,NaN,NaN,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000


olist_sellers_dataset.csv


,seller_id,seller_zip_code_prefix,seller_city,seller_state
count,3095,3095.000000,3095,3095
unique,3095,NaN,611,23
top,3442f8959a84dea7ee197c632cb2df15,NaN,sao paulo,SP
freq,1,NaN,694,1849
mean,NaN,32291.059451,NaN,NaN
std,NaN,32713.453830,NaN,NaN
min,NaN,1001.000000,NaN,NaN
25%,NaN,7093.500000,NaN,NaN
50%,NaN,14940.000000,NaN,NaN
75%,NaN,64552.500000,NaN,NaN


product_category_name_translation.csv


,product_category_name,product_category_name_english
count,71,71
unique,71,71
top,beleza_saude,health_beauty
freq,1,1


### Sample data
Here is the sample data to understand the overview of existing data these values can change evey time we run the data difing cell above

In [8]:
for item in samples_f:
  print(item["File_Name"])
  display(item["Samples"])

olist_customers_dataset.csv


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
77613,eaf693be16687cdaafc2b059b938efdd,ab126864b391ef4fdf42ed349a82acdc,14160,sertaozinho,SP
91906,810097a2f77b7dd3b52032ed9d7d3ca3,fc7b10310fe6110e2a4a967fe04c39a7,36576,porto firme,MG
20090,b44d67d74c2dab6a4597359b1edd75d8,f532b515dbddb3fcc2c63367244dac4b,59067,natal,RN
45959,e40736c0b4e32e738796a5fc57234dad,ead394416d4f85207932cc124eb145e2,5572,sao paulo,SP
8790,e165d4c6a0d945a813e8a3bf42faa500,4ef21f52dd7c061a646005b11918cbd3,60824,fortaleza,CE
27307,19f14bc6e61673618b16b2e68b523ab3,6b03072644d626499aaf5cf2347f72c2,4117,sao paulo,SP
82924,ad8357e9c4b9a78d08ec2bddcbf2dac9,518199e5f36305bffce18e38aa2642aa,13088,campinas,SP
6566,c863ee474411523071aa3d34080d1f24,02c0e149593ec363ed327a4651128d49,70747,brasilia,DF
77430,8c2db8ebb47d738813b679a5fde0634e,22be6b0c4ec4b916766bdacf188dcdad,5657,sao paulo,SP
53150,cfed44a643330d0984539ec92412055f,203be24e6117b285f1a0e57eab8d4b41,95070,caxias do sul,RS


olist_geolocation_dataset.csv


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
809253,75901,-17.792583,-50.926439,rio verde,GO
767170,66670,-1.341653,-48.447119,belem,PA
502828,27140,-22.477821,-43.818271,barra do pirai,RJ
939008,90010,-30.036174,-51.229610,porto alegre,RS
633582,37550,-22.226777,-45.935398,pouso alegre,MG
237941,9721,-23.715795,-46.547655,sao bernardo do campo,SP
706894,49160,-10.866255,-37.089429,nossa senhora do socorro,SE
168649,6654,-23.536643,-46.925938,itapevi,SP
519327,28890,-22.553807,-41.976959,rio das ostras,RJ
64751,3735,-23.511555,-46.522056,sao paulo,SP


olist_orders_dataset.csv


...

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
22893,edb0a8739ce7457ff4767a85d6aa92e2,fashion_roupa_feminina,35.0,436.0,1.0,700.0,20.0,20.0,20.0
22111,fb3d9b25b111fc87a868c07e66489cae,beleza_saude,52.0,2460.0,3.0,775.0,18.0,19.0,15.0
7707,79ad39409fa4bcc36b4b7f734c79c2a2,fashion_bolsas_e_acessorios,59.0,874.0,3.0,200.0,20.0,10.0,20.0
19836,741ab0ea9712d54a420f0a347ce71613,relogios_presentes,40.0,266.0,1.0,200.0,18.0,14.0,17.0
7623,f7b6ca5735bfd052e73fa0b776070112,eletronicos,59.0,694.0,2.0,175.0,21.0,2.0,15.0
2401,e855e71e84e8f333d5f82f54e22fda48,fashion_bolsas_e_acessorios,14.0,115.0,2.0,400.0,20.0,20.0,20.0
20754,911e953e7bd4d3a93bcc0354e12d2b96,automotivo,56.0,1128.0,3.0,450.0,18.0,6.0,12.0
15618,b1dc70b2d3b0d42b355529c4026a35a7,eletronicos,58.0,256.0,3.0,300.0,16.0,5.0,20.0
31862,f7f75f7339e0fd5af137094d59f44890,cool_stuff,58.0,1582.0,1.0,2000.0,28.0,36.0,27.0
13793,925e74f05a815d63be377531002e29b3,cama_mesa_banho,49.0,104.0,1.0,1500.0,36.0,14.0,36.0


olist_sellers_dataset.csv


,seller_id,seller_zip_code_prefix,seller_city,seller_state
2996,1900267e848ceeba8fa32d80c1a5f5a8,14940,ibitinga,SP
2243,3361277dc30b7cccdb0c286b24219756,87015,maringa,PR
147,0ade5cc4a305ed7093a23f68ca47519c,81030,curitiba,PR
856,2646baaf662d4d92ac48f047e35db92d,3303,sao paulo,SP
174,623ad9d0c1e14bb5464a8c72bff7b4af,35530,claudio,MG
2401,7202e2ba20579a9bd1acb29e61fe71f6,14401,franca,SP
56,7d294cf9a6a69dc6a0c3e540fff0e039,88390,barra velha,SC
1428,2c43fb513632d29b3b58df74816f1b06,30421,belo horizonte,MG
2058,7e3f87d16fb353f408d467e74fbd8014,4809,sao paulo,SP
2380,1703bc09972dab9782e7a9194943b69f,37165,minas gerais,MG


product_category_name_translation.csv


,product_category_name,product_category_name_english
23,malas_acessorios,luggage_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
19,consoles_games,consoles_games
20,audio,audio
48,livros_interesse_geral,books_general_interest
1,informatica_acessorios,computers_accessories
55,fashion_roupa_feminina,fashio_female_clothing
58,bebidas,drinks
26,moveis_cozinha_area_de_servico_jantar_e_jardim,kitchen_dining_laundry_garden_furniture


# Cleaning the data
By gathering all the data profile and info. The only process before we work on the data remained is cleaning it.

### Goals for now
* Check duplicates before dropping anything.
* why is customers id looks encryped in customers table and also why there are two keys one uniqe and another normal
* Found there are some zip_code_prefix with 4 digit values which is not possible replace it with original or check if there is 0 infront of those codes
* Fix types. Dates become real datetimes, and zip codes become zero-padded text.
* Orders. It checks the null delivery dates against order_status, and flags delivered orders with no delivery date. It also adds delivery_days, delivery_delay_days and is_late.
* why are the customers id and orders id matching did all the customers ordered only single time? *** wtf ***
* What is shipping limit date and order item id denotes why the ids looks encrypted
* what is freight value?
* How do the payment table work?
* review creation vs review answered date
* Nan Review titles should be converted to empty or ""?
* why there are duplicate review ids?
* Reviews. It removes repeated review_ids and adds a has_comment flag.
* Products. It fixes the lenght typos, fills the missing category with "unknown", and merges in the English names.
*Geolocation. It drops coordinates outside Brazil and collapses to one row per zip.
* Text and sanity checks. City names are lowercased and stripped of accents. It also checks payments and prices.
* Orphan-key checks between related tables.
* Final null check, then export to a Cleaned folder next to your Data folder.

### Analyzing the Files and noting down the key take aways
First step is to **keep the actual DataFrames** in a dict.
Do not `pd.concat` these tables into one master file: they have different columns and meanings. Keep them as separate tables (star schema) for Power BI.

In [9]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

DATA  = Path(r"X:\Powerbi\Projects\Brazillian_ecommerce_analysis\Data")
CLEAN = DATA.parent / "Cleaned"
CLEAN.mkdir(exist_ok=True)

short = {
    "olist_customers_dataset": "customers",
    "olist_geolocation_dataset": "geolocation",
    "olist_orders_dataset": "orders",
    "olist_order_items_dataset": "order_items",
    "olist_order_payments_dataset": "payments",
    "olist_order_reviews_dataset": "reviews",
    "olist_products_dataset": "products",
    "olist_sellers_dataset": "sellers",
    "product_category_name_translation": "category_translation",
}

zip_cols = {c: str for c in ["customer_zip_code_prefix",
                             "seller_zip_code_prefix",
                             "geolocation_zip_code_prefix"]}

d = {short[p.stem]: pd.read_csv(p, dtype=zip_cols)
     for p in DATA.glob("*.csv") if p.stem in short}

display({k: v.shape for k, v in d.items()})

{'customers': (99441, 5),
 'geolocation': (1000163, 5),
 'orders': (99441, 8),
 'order_items': (112650, 7),
 'payments': (103886, 5),
 'reviews': (99224, 7),
 'products': (32951, 9),
 'sellers': (3095, 4),
 'category_translation': (71, 2)}

## Duplicate check
Look before dropping anything. For each table this prints fully identical rows and, where a primary key exists, repeated key values.

In [10]:
keys = {"customers": "customer_id", "orders": "order_id", "reviews": "review_id",
        "products": "product_id", "sellers": "seller_id"}

for name, df in d.items():
    line = f"{name:22s} identical rows: {df.duplicated().sum():>7,}"
    if name in keys:
        line += f" | repeated {keys[name]}: {df[keys[name]].duplicated().sum():,}"
    print(line)

customers              identical rows:       0 | repeated customer_id: 0
geolocation            identical rows: 261,831
orders                 identical rows:       0 | repeated order_id: 0
order_items            identical rows:       0
payments               identical rows:       0
reviews                identical rows:       0 | repeated review_id: 814
products               identical rows:       0 | repeated product_id: 0
sellers                identical rows:       0 | repeated seller_id: 0
category_translation   identical rows:       0


## Understand the review duplicates before deciding
`review_id` repeats in this dataset. These three numbers tell you whether the same review is shared across several orders or whether rows are exact copies.
- If `(review_id, order_id)` repeats are near zero, one review covers several orders.
- If `order_id` repeats are above zero, some orders have more than one review.

In [11]:
r = d["reviews"]
print("repeated review_id            :", r["review_id"].duplicated().sum())
print("repeated (review_id, order_id):", r.duplicated(["review_id", "order_id"]).sum())
print("orders with more than 1 review:", r["order_id"].duplicated().sum())

repeated review_id            : 814
repeated (review_id, order_id): 0
orders with more than 1 review: 551


In [12]:
def zip5(s):
    return (s.astype("string").str.strip()
             .str.replace(r"\.0$", "", regex=True)
             .str.zfill(5))

## Zip codes: diagnose, then fix
Brazilian zip prefixes have 5 digits. A 4-digit value is a code whose leading zero was lost (such as `01046` saved as `1046`).
First confirm the theory: short codes should belong almost entirely to São Paulo (`SP`), whose codes start with 0.

In [13]:
c = d["customers"]
short_zip = c[c["customer_zip_code_prefix"].astype("string").str.strip().str.len() == 4]
print("4-digit zips:", len(short_zip))
print(short_zip["customer_state"].value_counts().head())

4-digit zips: 0
Series([], Name: count, dtype: int64)


In [14]:
for name, col in [("customers", "customer_zip_code_prefix"),
                  ("sellers", "seller_zip_code_prefix"),
                  ("geolocation", "geolocation_zip_code_prefix")]:
    d[name][col] = zip5(d[name][col])

print({n: d[n][c].str.len().value_counts().to_dict()
       for n, c in [("customers", "customer_zip_code_prefix"),
                    ("sellers", "seller_zip_code_prefix"),
                    ("geolocation", "geolocation_zip_code_prefix")]})

{'customers': {np.int64(5): 99441}, 'sellers': {np.int64(5): 3095}, 'geolocation': {np.int64(5): 1000163}}


## Convert text columns to real datetimes
`read_csv` loads dates as text. `errors="coerce"` turns anything unparseable into `NaT` (missing date) instead of crashing.

In [15]:
date_cols = {
    "orders": ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
               "order_delivered_customer_date", "order_estimated_delivery_date"],
    "order_items": ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}
for name, cols in date_cols.items():
    for col in cols:
        d[name][col] = pd.to_datetime(d[name][col], errors="coerce")

d["orders"].dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

## Orders: do the null dates make sense?
A missing delivery date is expected for canceled or in-transit orders. A **delivered** order without a delivery date is a genuine data problem.

In [16]:
orders = d["orders"]

status_check = (orders.assign(no_delivery_date=orders["order_delivered_customer_date"].isna())
                      .groupby("order_status")["no_delivery_date"].agg(no_delivery_date="sum", orders="size"))
display(status_check)

bad = orders[(orders["order_status"] == "delivered") & orders["order_delivered_customer_date"].isna()]
print("delivered but missing delivery date:", len(bad))

,no_delivery_date,orders
order_status,,
approved,2,2
canceled,619,625
created,5,5
delivered,8,96478
invoiced,314,314
processing,301,301
shipped,1107,1107
unavailable,609,609


delivered but missing delivery date: 8


## Orders: derived columns for the dashboard
- `order_purchase_date`: date without time, used to link a Power BI date table.
- `delivery_days`: purchase to delivery.
- `delivery_delay_days`: positive means delivered after the promised date.
- `is_late`: 1 / 0, and left blank when the order was never delivered (no fake 0).

In [17]:
orders["order_purchase_date"] = orders["order_purchase_timestamp"].dt.normalize()

orders["delivery_days"] = (orders["order_delivered_customer_date"]
                           - orders["order_purchase_timestamp"]).dt.days.astype("Int16")
orders["delivery_delay_days"] = (orders["order_delivered_customer_date"]
                                 - orders["order_estimated_delivery_date"]).dt.days.astype("Int16")

delay = orders["delivery_delay_days"]
orders["is_late"] = (delay > 0).astype("Int8").where(delay.notna())

# logical consistency: delivery should not come before purchase
print("delivered before purchase:",
      (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]).sum())

d["orders"] = orders

delivered before purchase: 0


## Orders: coverage and orders without items
Two checks that affect how you build measures later:
- orders with no rows in `order_items` (usually canceled/unavailable), so they carry no revenue
- orders per month, since the first and last months are often incomplete

In [18]:
no_items = ~d["orders"]["order_id"].isin(d["order_items"]["order_id"])
print("orders with no items:", no_items.sum())
display(d["orders"][no_items]["order_status"].value_counts())

monthly = d["orders"].groupby(d["orders"]["order_purchase_timestamp"].dt.to_period("M")).size()
display(monthly)

orders with no items: 775


order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, dtype: int64

## Customers: two keys, one person
`customer_id` is created per order. `customer_unique_id` identifies the person. This shows how many people ordered more than once.

In [19]:
co = d["orders"].merge(d["customers"], on="customer_id", how="left")
per_person = co.groupby("customer_unique_id")["order_id"].nunique()

print("distinct customer_id        :", d["customers"]["customer_id"].nunique())
print("distinct customer_unique_id :", d["customers"]["customer_unique_id"].nunique())
print("share of repeat customers   :", round((per_person > 1).mean() * 100, 2), "%")

distinct customer_id        : 99441
distinct customer_unique_id : 96096
share of repeat customers   : 3.12 %


## Reviews
1. Replace line breaks inside comments with spaces, so the CSV imports cleanly in Power Query.
2. Add `has_comment` **before** anything else touches the text (nulls stay null).
3. Keep one review per order, the most recently answered.
4. Add `review_response_hours` (survey sent to answer submitted).

In [20]:
rev = d["reviews"].copy()

for col in ["review_comment_title", "review_comment_message"]:
    rev[col] = rev[col].str.replace(r"[\r\n]+", " ", regex=True).str.strip()

rev["has_comment"] = rev["review_comment_message"].notna().astype(int)

before = len(rev)
rev = rev.sort_values("review_answer_timestamp").drop_duplicates(subset="order_id", keep="last")
print(f"reviews: {before:,} -> {len(rev):,}")

rev["review_response_hours"] = (rev["review_answer_timestamp"]
                                - rev["review_creation_date"]).dt.total_seconds() / 3600

print("answered before survey was sent:", (rev["review_response_hours"] < 0).sum())
print("scores outside 1-5             :", (~rev["review_score"].between(1, 5)).sum())

d["reviews"] = rev

reviews: 99,224 -> 98,673
answered before survey was sent: 0
scores outside 1-5             : 0


## Products
- Fix the `lenght` typos.
- Missing category becomes `"unknown"`.
- Merge the English category name. Two categories are missing from the translation table, so they are mapped by hand.
- Rows with missing weight/size are **kept** (dropping them would orphan rows in `order_items`).

In [21]:
prod = d["products"].rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
})
prod["product_category_name"] = prod["product_category_name"].fillna("unknown")

n_before = len(prod)
prod = prod.merge(d["category_translation"], on="product_category_name", how="left")
assert len(prod) == n_before, "merge duplicated rows: translation table has repeated keys"

missing_tr = prod.loc[prod["product_category_name_english"].isna()
                      & (prod["product_category_name"] != "unknown"), "product_category_name"].unique()
print("categories without a translation:", list(missing_tr))

manual = {"pc_gamer": "pc_gamer",
          "portateis_cozinha_e_preparadores_de_alimentos": "portable_kitchen_food_preparers"}
prod["product_category_name_english"] = (prod["product_category_name_english"]
                                         .fillna(prod["product_category_name"].map(manual))
                                         .fillna(prod["product_category_name"]))

print("weight == 0   :", (prod["product_weight_g"] == 0).sum())
print("missing weight:", prod["product_weight_g"].isna().sum())

d["products"] = prod

categories without a translation: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
weight == 0   : 4
missing weight: 2


## Geolocation
About a million rows with many repeats per zip. Steps: drop coordinates outside Brazil, standardise city text, then collapse to one row per zip (mean coordinates, most common city), so joins don't multiply rows.

In [22]:
def clean_text(s):
    """lowercase, trim, remove accents (sao paulo = São Paulo = SAO PAULO)"""
    s = s.astype(str).str.strip().str.lower()
    return s.map(lambda x: unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode())

geo = d["geolocation"].copy()
n0 = len(geo)
geo = geo[geo["geolocation_lat"].between(-35, 6) & geo["geolocation_lng"].between(-75, -34)].copy()
print("rows outside Brazil removed:", n0 - len(geo))

geo["geolocation_city"] = clean_text(geo["geolocation_city"])

d["geolocation"] = (geo.groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(geolocation_lat=("geolocation_lat", "mean"),
         geolocation_lng=("geolocation_lng", "mean"),
         geolocation_city=("geolocation_city", lambda s: s.mode().iat[0]),
         geolocation_state=("geolocation_state", "first")))

print("geolocation rows now:", len(d["geolocation"]))

rows outside Brazil removed: 40
geolocation rows now: 19010


## Customers and sellers: text standardisation
Same `clean_text` helper for city names. State codes are trimmed and upper-cased.

In [23]:
for name, city_col, state_col in [("customers", "customer_city", "customer_state"),
                                  ("sellers", "seller_city", "seller_state")]:
    d[name][city_col]  = clean_text(d[name][city_col])
    d[name][state_col] = d[name][state_col].str.strip().str.upper()

d["customers"]["customer_state"].value_counts().head()

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
Name: count, dtype: int64

## Payments and order items: sanity checks
These print counts instead of stopping the notebook, so you can decide what to do with anything unusual.

In [24]:
pay = d["payments"]
display(pay["payment_type"].value_counts())
print("payment_value == 0:", (pay["payment_value"] == 0).sum())
print("installments == 0 :", (pay["payment_installments"] == 0).sum())

items = d["order_items"]
print("price <= 0        :", (items["price"] <= 0).sum())
print("freight < 0       :", (items["freight_value"] < 0).sum())

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

payment_value == 0: 9
installments == 0 : 2
price <= 0        : 0
freight < 0       : 0


## Referential integrity (orphan keys)
Every child key should exist in its parent table. Non-zero counts need investigating before you build the Power BI model.

In [25]:
def orphans(child, col, parent, pcol=None):
    pcol = pcol or col
    n = (~d[child][col].isin(d[parent][pcol])).sum()
    print(f"{child}.{col:28s} not in {parent}: {n:,}")

orphans("orders", "customer_id", "customers")
orphans("order_items", "order_id", "orders")
orphans("order_items", "product_id", "products")
orphans("order_items", "seller_id", "sellers")
orphans("payments", "order_id", "orders")
orphans("reviews", "order_id", "orders")
orphans("customers", "customer_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix")
orphans("sellers", "seller_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix")

orders.customer_id                  not in customers: 0
order_items.order_id                     not in orders: 0
order_items.product_id                   not in products: 0
order_items.seller_id                    not in sellers: 0
payments.order_id                     not in orders: 0
reviews.order_id                     not in orders: 0
customers.customer_zip_code_prefix     not in geolocation: 279
sellers.seller_zip_code_prefix       not in geolocation: 7
